<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-05-deterministic-mini-agent/notebook.ipynb)


# Session 5 — A deterministic mini-agent

**Goal:** complete a tool-calling loop with a trace receipt: a loop budget, a repeated call caught, and a safe termination. *Thread: loop engineering.*

Every cell runs offline on `FakeLLM`. Nothing here calls a provider or the network.


In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

from bootcamp_agent.checks import check, review


✅ Python 3.13 (need >= 3.11)
⚠️  kernel is the repo .venv  -> pick the .venv kernel in Jupyter, or start it with: uv run jupyter lab
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


## 1. The loop you already have

`answer_question` is a loop with three exits already designed: refuse when retrieval is empty, retry once on a broken contract, refuse again if the retry fails. The trace is what it did, in order.

In [ ]:
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM

documents = load_corpus(CORPUS_DIR)
question = "What defenses help against prompt injection?"
result = answer_question(question, documents, FakeLLM())
for event in result.trace:
    print(f"[{event.kind}] {event.detail}")


[retrieve] top_k=3 -> [('prompt-injection', 1), ('prompt-injection', 0), ('structured-outputs', 2)]
[llm_call] attempt 1: 121 chars
[decision] answered with citations []


## 2. Exercise: the budget, visible in the trace

**Context.** `answer_question` takes `max_tool_calls`. With this corpus the direct path rarely needs a tool; the point is that the bound exists and the trace shows it.

**Instructions.**

1. **Run the cell.** Both budgets are already written: the same question and corpus, once at 3 and once at 1.
2. **Read the two lists.** Count the `tool_call` events in each, against the budget it was given.
3. **Run the check.** It confirms neither trace exceeded its own budget, and that both end in a `decision` — the loop chose to stop, rather than running out of plan.

In [ ]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# To stand above it: stop on a budget you can see in the trace, not one buried in a constant.
# Change it, re-run the check cell below, and keep what you learn.
# ---------------------------------------------------------------------
traces = {
    3: [e.kind for e in answer_question(question, documents, FakeLLM(), max_tool_calls=3).trace],
    1: [e.kind for e in answer_question(question, documents, FakeLLM(), max_tool_calls=1).trace],
}
for budget, kinds in traces.items():
    print(f"budget={budget}: {kinds}")


budget=3: ['retrieve', 'llm_call', 'decision']
budget=1: ['retrieve', 'llm_call', 'decision']


**Expected output** (yours may differ in wording, not in shape):

```
budget=3: ['retrieve', 'llm_call', 'decision']
budget=1: ['retrieve', 'llm_call', 'decision']
✅ ch05-e1 passed
```

In [ ]:
check("ch05-e1", traces)

✅ ch05-e1 passed


True

## 3. The tools, and a plan over them

Your loop needs something to call. These are session 4's two tools with their contracts intact, plus a **plan**: the sequence of calls a model would have chosen, written down instead. A scripted plan makes every exit reachable on purpose, so the loop is testable without a model in it.

In [ ]:
from bootcamp_agent.tools import ToolError

# Yesterday's two tools, unchanged in contract. The rate table is pinned so this
# notebook never touches the network, and the ids join on one line so a receipt
# prints on one screen.
RATES = {"USD": {"EUR": 0.92, "BRL": 5.40}}


def list_documents(tag: str | None = None) -> str:
    known = {t for doc in documents for t in doc.tags}
    if tag is None:
        return ", ".join(doc.doc_id for doc in documents)
    if not tag.strip():
        raise ToolError("list_documents: 'tag' must be non-empty when given")
    if tag not in known:
        raise ToolError(f"list_documents: unknown tag {tag!r}; valid tags: {sorted(known)}")
    return ", ".join(doc.doc_id for doc in documents if tag in doc.tags)


def convert_currency(amount: float, source: str, target: str) -> str:
    rates = RATES.get(source, {})
    if target not in rates:
        raise ToolError(f"convert_currency: no rate {source}->{target}; known: {sorted(rates)}")
    return f"{amount} {source} = {amount * rates[target]:.2f} {target}"


tools = {"list_documents": list_documents, "convert_currency": convert_currency}
plan = [
    {"tool": "list_documents", "args": {"tag": "retrieval"}},
    {"tool": "convert_currency", "args": {"amount": 100, "source": "USD", "target": "EUR"}},
    {"tool": "answer", "args": {"text": "rag-basics covers retrieval, and 100 USD is 92.00 EUR."}},
]
for step in plan:
    print(f"{step['tool']:18} {step['args']}")


list_documents     {'tag': 'retrieval'}
convert_currency   {'amount': 100, 'source': 'USD', 'target': 'EUR'}
answer             {'text': 'rag-basics covers retrieval, and 100 USD is 92.00 EUR.'}


## 4. Exercise: `run_loop`, and its four exits

**Context.** The loop executes one planned call at a time and returns a receipt. Every run ends in exactly one of four designed states — never in a traceback.

| `stopped_because` | When | `answer` | `refusal` |
|---|---|---|---|
| `answered` | the step's tool is `answer` | its `args['text']` | `None` |
| `repeated_call` | this call equals the one before it | `None` | why |
| `budget` | `budget` calls already recorded, or the plan ran out | `None` | why |
| `tool_error` | the tool raised `ToolError` | `None` | why, naming the tool |

`steps` records executed **tool calls** only, one `{"tool", "args", "result"}` dict each. The `answer` step is a decision, not a call, so it is never a step.

**How to do it, in five steps.**

1. **Run the "Get started" cell** below. It writes one exit of its own, so you can see the shape before you write four.
2. **In the challenge cell, find `exit 2`.** Each exit is already there as two commented lines with a `___` in them. You uncomment and fill the blank.
3. **One exit at a time.** Uncomment it, replace the `___`, run the cell. The last line prints where the loop stopped.
4. **Write the refusal as a sentence.** Every stop that is not `answered` fills `refusal`; a caller who reads only the receipt has to know why it ended.
5. **Run the check.** It names the scenario that is still wrong — the repeated plan, the budget, or the tool that raised.

In [ ]:
# GET STARTED WITH THE LOOP. This cell runs as it is. Nothing here is marked.
#
# TIPS
#   1. Write the EXITS first, the body afterwards. A loop whose only ending is
#      success will spin, spend, or hand a traceback to whatever called it.
#   2. One exit at a time. Uncomment one, run the cell, read what it prints.
#   3. Every stop that is not `answered` writes a sentence into `refusal`.
#      Somebody reads it. "stopped" alone tells them nothing.
#   4. Stuck? Ask the coach (the calls at the end of this cell).

# ONE WORKED EXAMPLE, the same shape as the exits you write — and NOT one of them.
# It stops on a rule of its own: too many words asked for at once.
def run_until_too_long(words: list[str], limit: int = 3) -> dict:
    taken: list[str] = []
    for word in words:
        if len(taken) >= limit:                      # the exit, checked BEFORE the work
            return receipt_like(taken, "too_long", refusal=f"stopped: {limit} words is the limit")
        taken.append(word)
    return receipt_like(taken, "finished", answer=" ".join(taken))


def receipt_like(taken, stopped_because, answer=None, refusal=None) -> dict:
    """The same four keys `receipt` has. Yours is given to you below."""
    return {"steps": taken, "stopped_because": stopped_because, "answer": answer, "refusal": refusal}


for words in (["a", "short", "one"], ["this", "one", "is", "far", "too", "long"]):
    outcome = run_until_too_long(words)
    print(f'{outcome["stopped_because"]:10} {outcome["answer"] or outcome["refusal"]}')

# Read what that example does, because your four exits do the same three things:
#   check the exit BEFORE doing the work · return the receipt · say why in a sentence.

# ASK THE COURSE. This question runs. Then uncomment ONE line at a time.
from bootcamp_agent.coach import coach
coach("write the exits before the body of a loop", top_k=1, max_chars=700)
#coach("the budget belongs to the app not the plan", top_k=1, max_chars=700)
#coach("repetition is the cheap spin detector", top_k=1, max_chars=700)
#coach("a refusal is written for a reader", top_k=1, max_chars=700)


finished   a short one
too_long   stopped: 3 words is the limit
--- Follow along: today's class  [unit1/session-05-deterministic-mini-agent/follow-along]
| Part | What | Time |
|---|---|---|
| [0](#0-before-we-start) | Before we start: update | 3 min |
| [1](#1-a-loop-you-already-have) | A loop you already have | 10 min |
| [2](#2-write-the-exits-before-the-body) | Write the exits before the body | 10 min |
| [3](#3-live-demo-the-coach-in-a-chat) | Live demo: the coach in a chat | 20 min |
| [4](#4-your-exercise) | Your exercise: the loop, and its four exits | 45 min |
| [5](#5-the-weekly-challenge) | The weekly challenge: a bot that refuses well | 5 min |
| [6](#6-optional-put-it-on-telegram) | Optional: put it on Telegram | at home |

## 0. Before we start

From your course folder:

```bash
git pull
uv run jupyter lab
```


In [ ]:
# ---------------------------------------------------------------------
# THE ONE CELL IN THIS SESSION THAT DOES NOT RUN AS SHIPPED.
# The others are written: run them and you have 100 of 200 marks.
# This is the rest. It is the session's point, so it is the one you write.
#
# HOW: run the "Get started" cell above first — it shows one worked exit.
# Each exit below is already written as a commented line with a ___ in it.
# Uncomment one, replace the ___, run the cell, then run the check.
# ---------------------------------------------------------------------
from collections.abc import Callable


def receipt(steps, stopped_because, answer=None, refusal=None) -> dict:
    """The four keys, on every exit. Given to you; do not change the shape."""
    return {
        "steps": steps,
        "stopped_because": stopped_because,
        "answer": answer,
        "refusal": refusal,
    }


def run_loop(plan: list[dict], tools: dict[str, Callable], budget: int = 5) -> dict:
    steps: list[dict] = []
    previous = None
    for step in plan:
        name, args = step["tool"], step.get("args", {})

        if name == "answer":
            return receipt(steps, "answered", answer=args["text"])

        if (name, args) == previous:
            return receipt(
                steps,
                "repeated_call",
                refusal="Are we repeating ourselves? Try again."
            )

        if len(steps) >= budget:
            return receipt(
                steps,
                "budget",
                refusal="STOPPED: The call limit has been reached."
            )

        try:
          result = tools[name](**args)
        except ToolError as error:
            return receipt(
                steps,
                "tool_error",
                refusal=f"{name} failed: {error}"
              )

        steps.append({"tool": name, "args": args, "result": result})
        previous = (name, args)
    return receipt(steps, "budget", refusal="stopped: the plan ran out before an answer")


print(run_loop(plan, tools)["stopped_because"])


answered


**Expected output** (yours may differ in wording, not in shape):

```
answered
✅ ch05-e2 passed
```

In [ ]:
check("ch05-e2", run_loop)

✅ ch05-e2 passed


True

## 5. The receipt, read back

This is the artifact: what was called, with what arguments, what came back, and why the run ended. Nobody has to trust a summary of the run when they can read the run.

In [ ]:
lab = run_loop(plan, tools)
for index, step in enumerate(lab["steps"], 1):
    print(f"{index}. {step['tool']}({step['args']}) -> {step['result']}")
print(f"stopped_because={lab['stopped_because']!r}  answer={lab['answer']!r}")


1. list_documents({'tag': 'retrieval'}) -> rag-basics
2. convert_currency({'amount': 100, 'source': 'USD', 'target': 'EUR'}) -> 100 USD = 92.00 EUR
stopped_because='answered'  answer='rag-basics covers retrieval, and 100 USD is 92.00 EUR.'


## 6. Failure injection: a tool that starts refusing

A tool that works in the first cell and fails in the fourth is the normal case, not the exotic one: a rate limit, an expired token, an index rebuild. The loop must end with a refusal the caller can read.

Run this before you finish exit 4, and again after. The difference is the lesson.

In [ ]:
calls = {"n": 0}


def flaky_list_documents(tag: str | None = None) -> str:
    """Answers twice, then refuses. A real tool fails mid-run; this one fails on cue."""
    calls["n"] += 1
    if calls["n"] > 2:
        raise ToolError("list_documents: the corpus index went away mid-run")
    return list_documents(tag)


flaky_plan = [
    {"tool": "list_documents", "args": {"tag": "retrieval"}},
    {"tool": "list_documents", "args": {"tag": "security"}},
    {"tool": "list_documents", "args": {"tag": "evaluation"}},
    {"tool": "answer", "args": {"text": "never reached"}},
]
try:
    injected = run_loop(flaky_plan, {"list_documents": flaky_list_documents})
    print(f"steps={len(injected['steps'])}  stopped_because={injected['stopped_because']!r}")
    print(f"refusal: {injected['refusal']}")
except ToolError as error:
    print(f"the error escaped the loop: {error}")
    print("that is the bug — exit 4 in run_loop turns it into a refusal")


steps=2  stopped_because='tool_error'
refusal: list_documents failed: list_documents: the corpus index went away mid-run


## Exit ticket

One thing that works, one thing that is unclear, your next action.

Homework: write the exit table for a loop you have built or used. If a row is empty, that loop is unfinished. Read `docs/guides/loop-engineering.md`.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch05")

ch05: 2/2 passed  ·  200/200 marks


True

## Weekly challenge (adds up to 500 to this session's score)

**The brief:** a bot whose four exits a stranger can see. The full brief is the
[weekly challenge page](https://gecko-academy.github.io/dev3pack-cohort-2026-09/unit1/session-05-deterministic-mini-agent/weekly-challenge), and
[demo 8](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/demos/08_the_weekly_challenge.ipynb) walks through one bot
start to finish. **This cell is where yours is graded.**

One function is the whole contract:

| Key it returns | What it holds |
|---|---|
| `stopped_because` | one of `answered` · `repeated_call` · `budget` · `tool_error` |
| `reply` | what the person reads. A refusal is a sentence, not a word |

`chat` is yours: a dict that survives between messages, so keep the count of
calls in it. The check scores out of 500, in five tiers of 100, and **100 is a
pass**. The score it prints is added to this session's score when you hand the
notebook in, so it moves you up the leaderboard. Skip it and you lose nothing.

**Hand it in:** write `respond` below, run the check cell, save, then
`uv run bootcamp submit ch05 --github <your-github-name> --push`. Already
submitted ch05? Add the bot, run the cell, and submit again. There is no limit.


In [ ]:
BUDGET = 3


def route(text: str) -> list[dict]:
    """Turn Ana's message into a deterministic tool plan."""
    lower = text.strip().lower()

    if lower.startswith("/page "):
        page_id = text.strip()[len("/page "):].strip()
        return [
            {
                "tool": "read_page",
                "args": {"page_id": page_id},
            },
            {
                "tool": "answer",
                "args": {
                    "text": "Use the approved page result above."
                },
            },
        ]

    if "taxi" in lower:
        words = lower.split()
        amounts = [
            word
            for word in words
            if word.replace(".", "", 1).isdigit()
        ]

        if amounts and "brl" in lower:
            return [
                {
                    "tool": "convert_currency",
                    "args": {
                        "amount": float(amounts[0]),
                        "source": "BRL",
                        "target": "EUR",
                    },
                },
                {
                    "tool": "lookup_policy",
                    "args": {"category": "taxi"},
                },
                {
                    "tool": "answer",
                    "args": {
                        "text": (
                            "Use the conversion result and taxi policy "
                            "to determine whether the expense is covered."
                        )
                    },
                },
            ]

        return [
            {
                "tool": "lookup_policy",
                "args": {"category": "taxi"},
            },
            {
                "tool": "answer",
                "args": {
                    "text": "The taxi policy is shown above."
                },
            },
        ]

    if "convert" in lower:
        words = lower.split()

        amounts = [
            word
            for word in words
            if word.replace(".", "", 1).isdigit()
        ]

        currencies = [
            word.upper()
            for word in words
            if len(word) == 3 and word.isalpha()
        ]

        if amounts and len(currencies) >= 2:
            return [
                {
                    "tool": "convert_currency",
                    "args": {
                        "amount": float(amounts[0]),
                        "source": currencies[0],
                        "target": currencies[1],
                    },
                },
                {
                    "tool": "answer",
                    "args": {
                        "text": "The currency conversion is shown above."
                    },
                },
            ]

    for category in ("meals", "hotel", "taxi"):
        if category in lower:
            return [
                {
                    "tool": "lookup_policy",
                    "args": {"category": category},
                },
                {
                    "tool": "answer",
                    "args": {
                        "text": "The policy result above answers the question."
                    },
                },
            ]

    category = lower.split()[-1] if lower else ""

    return [
        {
            "tool": "answer",
            "args": {"text": "I can help with meals, hotel, taxi, and currency questions."},
        }
    ]


def respond(text: str, chat: dict) -> dict:
    """One message in, one receipt out."""
    chat.setdefault("calls", 0)
    chat.setdefault("last", None)

    normalised = text.strip().lower()

    # A repeated request is rejected before it can spend a call.
    if normalised == chat["last"]:
        return {
            "stopped_because": "repeated_call",
            "reply": (
                "You just asked that. Same question, same answer — "
                "ask me something different."
            ),
        }

    chat["last"] = normalised

    # The application owns the conversation budget.
    if chat["calls"] >= BUDGET:
        return {
            "stopped_because": "budget",
            "reply": (
                f"You've reached the {BUDGET}-call budget. "
                "Come back later when your budget resets."
            ),
        }

    plan = route(text)

    tools = {
        "lookup_policy": lookup_policy,
        "read_page": read_page,
        "convert_currency": convert_currency,
    }

    try:
        # The deterministic loop owns tool execution.
        receipt = run_loop(
            plan,
            tools,
            budget=5,
        )

    except ToolError as error:
        chat["calls"] += 1

        return {
            "stopped_because": "tool_error",
            "reply": f"stopped: {error}",
        }

    except Exception as error:
        chat["calls"] += 1

        return {
            "stopped_because": "tool_error",
            "reply": f"stopped: {error}",
        }

    # An accepted request consumes one conversation-budget slot.
    chat["calls"] += 1

    if receipt["stopped_because"] == "answered":
        results = [
            str(step["result"])
            for step in receipt["steps"]
        ]

        if receipt.get("answer"):
            results.append(str(receipt["answer"]))

        reply = "\n".join(results).strip()

        return {
            "stopped_because": "answered",
            "reply": reply or "Answered.",
            "steps": receipt["steps"],
        }

    return {
        "stopped_because": receipt["stopped_because"],
        "reply": (
            receipt.get("refusal")
            or receipt.get("answer")
            or "The request could not be completed."
        ),
        "steps": receipt["steps"],
    }


respond.examples = [
    "how much for meals",
    "convert 900 BRL to EUR",
    "is 900 BRL for a taxi covered?",
]

respond.broken = "/page nothing-like-this"

In [ ]:
from bootcamp_agent.bonus import bonus
from bootcamp_agent.weekly import week1_bot  # noqa: F401 (registers the check)

bonus("week1-bot", respond)



   week 1 challenge: 500/500
     ✅ the four exits           every exit is reachable from outside, and each one says why
     ✅ a tool of your own       something that can refuse, and whose refusal reaches the reader
     ✅ the receipt is visible   the reader can see which exit they got, without asking
     ✅ the budget recovers      it refuses, and it says when to come back — then it does
     ✅ your own loop            `run_loop` from ch05-e2 is behind it, walking a plan
✅ bonus week1-bot passed — above the floor.


True